In [8]:
from pathlib import Path
import pandas as pd
from ena_context import ExperimentContext

CONTEXTS_PATH = Path("../output/contexts.jsonl")
ACCESSIONS_PATH = Path("../metadata_analysis/v2_lung/datasets.csv")

with open(CONTEXTS_PATH) as f:
    contexts = [ExperimentContext.model_validate_json(line) for line in f if line.strip()]

source_accessions = pd.read_csv(ACCESSIONS_PATH)["srx_accession"].tolist()
print(f"Source accessions:  {len(source_accessions)}")
print(f"Loaded contexts:    {len(contexts)}")

Source accessions:  801
Loaded contexts:    801


## 1. Basic counts

In [9]:
loaded_accessions = [ctx.accession for ctx in contexts]

missing = set(source_accessions) - set(loaded_accessions)
extra   = set(loaded_accessions) - set(source_accessions)
dupes   = [a for a in loaded_accessions if loaded_accessions.count(a) > 1]

print(f"Missing from contexts: {len(missing)}")
print(f"Extra in contexts:     {len(extra)}")
print(f"Duplicates:            {len(set(dupes))}")

if missing:
    print("\nMissing accessions:", sorted(missing))

Missing from contexts: 0
Extra in contexts:     0
Duplicates:            0


## 2. Field coverage

In [10]:
n = len(contexts)

fields = {
    "studyDescription":              sum(1 for c in contexts if c.study and c.study.studyDescription),
    "pubmedAbstract":                sum(1 for c in contexts if c.study and c.study.pubmedAbstract),
    "biological.tissueType":         sum(1 for c in contexts if c.biological.tissueType),
    "biological.cellType":           sum(1 for c in contexts if c.biological.cellType),
    "biological.sampleAttributes":   sum(1 for c in contexts if c.biological.sampleAttributes),
    "technical.libraryStrategy":     sum(1 for c in contexts if c.technical.libraryStrategy),
    "technical.libraryConstructionProtocol": sum(1 for c in contexts if c.technical.libraryConstructionProtocol),
}

coverage = pd.DataFrame({
    "field": fields.keys(),
    "populated": fields.values(),
    "missing": [n - v for v in fields.values()],
    "pct": [f"{v/n*100:.1f}%" for v in fields.values()],
})
coverage

,field,populated,missing,pct
0,studyDescription,801,0,100.0%
1,pubmedAbstract,530,271,66.2%
2,biological.tissueType,621,180,77.5%
3,biological.cellType,285,516,35.6%
4,biological.sampleAttributes,755,46,94.3%
5,technical.libraryStrategy,801,0,100.0%
6,technical.libraryConstructionProtocol,647,154,80.8%


## 3. Warnings

In [11]:
warned = [(ctx.accession, w) for ctx in contexts for w in ctx.warnings]

print(f"Records with warnings: {sum(1 for c in contexts if c.warnings)} / {n}")
print(f"Total warning entries: {len(warned)}\n")

warning_types = pd.Series([w for _, w in warned]).value_counts()
print("Warning type breakdown:")
print(warning_types.to_string())

if warned:
    print("\nAffected accessions:")
    for acc, w in warned:
        print(f"  {acc}: {w}")

Records with warnings: 46 / 801
Total warning entries: 46

Warning type breakdown:
sample_xml_failed:curl failed for 'https://www.ebi.ac.uk/ena/browser/api/xml/SAMN40627613' (exit 56):     1
sample_xml_failed:curl failed for 'https://www.ebi.ac.uk/ena/browser/api/xml/SAMN46166561' (exit 56):     1
sample_xml_failed:curl failed for 'https://www.ebi.ac.uk/ena/browser/api/xml/SAMN46166613' (exit 56):     1
sample_xml_failed:curl failed for 'https://www.ebi.ac.uk/ena/browser/api/xml/SAMN46166612' (exit 56):     1
sample_xml_failed:curl failed for 'https://www.ebi.ac.uk/ena/browser/api/xml/SAMN46166607' (exit 56):     1
sample_xml_failed:curl failed for 'https://www.ebi.ac.uk/ena/browser/api/xml/SAMN46166604' (exit 56):     1
sample_xml_failed:curl failed for 'https://www.ebi.ac.uk/ena/browser/api/xml/SAMN46166603' (exit 56):     1
sample_xml_failed:curl failed for 'https://www.ebi.ac.uk/ena/browser/api/xml/SAMN46166610' (exit 56):     1
sample_xml_failed:curl failed for 'https://www.ebi.ac

## 4. Distributions

In [12]:
print("Library strategy:")
print(pd.Series([c.technical.libraryStrategy for c in contexts]).value_counts().to_string())

print("\nSpecies:")
print(pd.Series([c.biological.scientificName for c in contexts]).value_counts().to_string())

print("\nTissue type:")
print(pd.Series([c.biological.tissueType for c in contexts]).value_counts().head(20).to_string())

Library strategy:
RNA-Seq    770
OTHER       31

Species:
Homo sapiens    751
Mus musculus      4

Tissue type:
Lung                                                                                                                                                                                              167
lung                                                                                                                                                                                              142
Whole Lung Dissociate                                                                                                                                                                              49
More_fibrotic;Lung                                                                                                                                                                                 36
Less_fibrotic;Lung                                                                              

## 5. Spot checks

In [13]:
def print_ctx(ctx: ExperimentContext) -> None:
    print(f"Accession:          {ctx.accession}")
    print(f"Experiment title:   {ctx.experimentTitle}")
    print(f"Species:            {ctx.biological.scientificName}")
    print(f"Tissue:             {ctx.biological.tissueType}")
    print(f"Library strategy:   {ctx.technical.libraryStrategy}")
    print(f"Study description:  {(ctx.study.studyDescription or '')[:200]}")
    print(f"GEO accession:      {ctx.study.geoAccession}")
    print(f"PubMed abstract:    {(ctx.study.pubmedAbstract or '')[:200]}")
    print(f"Warnings:           {ctx.warnings}")
    print()

# first 3 records
for ctx in contexts[:3]:
    print_ctx(ctx)

Accession:          ERX11662371
Experiment title:   Illumina NovaSeq 6000 sequencing: Single cell RNA-seq atlas of human NSCLC lesions and non-involved tissue
Species:            Homo sapiens
Tissue:             None
Library strategy:   RNA-Seq
Study description:  We performed single cell RNA sequencing (scRNA-seq) of NSCLC tumours and matched, adjacent, non-involved lung tissue from 24 patients. The data set is composed of approximately 900,000 cells from two 
GEO accession:      None
PubMed abstract:    
Warnings:           []

Accession:          SRX19233503
Experiment title:   NextSeq 500 sequencing: GSM7016385: Replicate 3, scRNAseq Homo sapiens RNA-Seq
Species:            Homo sapiens
Tissue:             Distal Thrombus removed from pulmonary artery of CTEPH patient
Library strategy:   RNA-Seq
Study description:  Our current understanding of CTEPH pathobiology is primarily derived from cell-based studies limited by the use of specific cell markers or phenotypic modulation in cell

In [14]:
# records missing pubmedAbstract
no_abstract = [c for c in contexts if not (c.study and c.study.pubmedAbstract)]
print(f"Records without pubmedAbstract: {len(no_abstract)}\n")
for ctx in no_abstract[:5]:
    print_ctx(ctx)

Records without pubmedAbstract: 271

Accession:          ERX11662371
Experiment title:   Illumina NovaSeq 6000 sequencing: Single cell RNA-seq atlas of human NSCLC lesions and non-involved tissue
Species:            Homo sapiens
Tissue:             None
Library strategy:   RNA-Seq
Study description:  We performed single cell RNA sequencing (scRNA-seq) of NSCLC tumours and matched, adjacent, non-involved lung tissue from 24 patients. The data set is composed of approximately 900,000 cells from two 
GEO accession:      None
PubMed abstract:    
Warnings:           []

Accession:          ERX11662373
Experiment title:   Illumina NovaSeq 6000 sequencing: Single cell RNA-seq atlas of human NSCLC lesions and non-involved tissue
Species:            Homo sapiens
Tissue:             None
Library strategy:   RNA-Seq
Study description:  We performed single cell RNA sequencing (scRNA-seq) of NSCLC tumours and matched, adjacent, non-involved lung tissue from 24 patients. The data set is composed of

In [ ]:
# Look up context by accession

def get_context_by_accession(accession: str):
    """Return the ExperimentContext for a given accession, or None if not found."""
    return next((ctx for ctx in contexts if ctx.accession == accession), None)


accession_to_find = "SRX17412841"
ctx = get_context_by_accession(accession_to_find)
if ctx:
    print_ctx(ctx)
else:
    print(f"No context found for accession: {accession_to_find}")

Accession:          SRX17412841
Experiment title:   Illumina HiSeq 4000 sequencing: GSM4058964: 010I scRNAseq Homo sapiens RNA-Seq
Species:            Homo sapiens
Tissue:             Whole Lung Dissociate
Library strategy:   RNA-Seq
Study description:  Single Cell RNAseq of Whole Lung Dissociates from IPF, COPD and control lungs Overall design: Exploratory analysis of disease and healthy lung cell populations
GEO accession:      GSE136831
PubMed abstract:    Airway mucociliary regeneration and function are key players for airway defense and are impaired in chronic obstructive pulmonary disease (COPD). Using transcriptome analysis in COPD-derived bronchial
Warnings:           []



In [17]:
# Define the character threshold
char_limit = 300

# List for combined lengths
combined_lengths = [
    len((ctx.study.studyDescription or "")) + len((ctx.study.pubmedAbstract or ""))
    for ctx in contexts
]

# Count how many are below the threshold
below_limit = sum(length < char_limit for length in combined_lengths)
total = len(combined_lengths)

if total > 0:
    pct_below = 100 * below_limit / total
    print(f"{below_limit} of {total} records ({pct_below:.1f}%) have a combined studyDescription+pubmedAbstract length < {char_limit} chars")
else:
    print("No contexts to analyze.")

94 of 801 records (11.7%) have a combined studyDescription+pubmedAbstract length < 300 chars


In [22]:

def experiment_context_summary(accession: str):
    """Build a text string for an experiment accession containing title, species, tissue, library strategy,
    library prep method (technical), study description, and abstract (as available).
    """
    ctx = get_context_by_accession(accession)
    if not ctx:
        return f"No context found for accession: {accession}"

    parts = []

    # Experiment Title
    title = getattr(ctx, "title", None) or getattr(ctx.experiment, "title", None) if hasattr(ctx, "experiment") else None
    if title:
        parts.append(f"Title: {title}")

    # Species
    organism = None
    if hasattr(ctx, "sample") and ctx.sample:
        organism = getattr(ctx.sample, "scientificName", None) or getattr(ctx.sample, "organism", None)
    if organism:
        parts.append(f"Species: {organism}")

    # Tissue
    tissue = None
    if hasattr(ctx, "biological") and ctx.biological:
        tissue = getattr(ctx.biological, "tissueType", None)
    if tissue:
        parts.append(f"Tissue: {tissue}")

    # Library Strategy
    library_strategy = None
    if hasattr(ctx, "technical") and ctx.technical:
        library_strategy = getattr(ctx.technical, "libraryStrategy", None)
    if library_strategy:
        parts.append(f"Library strategy: {library_strategy}")

    # Library Preparation Method
    lib_prep = None
    if hasattr(ctx, "technical") and ctx.technical:
        # Try to get a main protocol, else fallbacks
        lib_prep = getattr(ctx.technical, "libraryConstructionProtocol", None) or getattr(ctx.technical, "libraryProtocol", None)
    if lib_prep:
        parts.append(f"Library prep: {lib_prep}")

    # Study Description
    study_desc = None
    if hasattr(ctx, "study") and ctx.study:
        study_desc = getattr(ctx.study, "studyDescription", None)
    if study_desc:
        parts.append(f"Description: {study_desc}")

    # Abstract
    abstract = None
    if hasattr(ctx, "study") and ctx.study:
        abstract = getattr(ctx.study, "pubmedAbstract", None)
    if abstract:
        parts.append(f"Abstract: {abstract}")

    if not parts:
        return "No descriptive fields found for this accession."

    return ". ".join(parts)



accession = "SRX17412841"
summary_text = experiment_context_summary(accession)
print(summary_text)



Tissue: Whole Lung Dissociate. Library strategy: RNA-Seq. Library prep: Thawed Cells were rinsed with 10% FBS passed through 40 uM strainer 10X Genomics 3 prime v2. Description: Single Cell RNAseq of Whole Lung Dissociates from IPF, COPD and control lungs Overall design: Exploratory analysis of disease and healthy lung cell populations. Abstract: Airway mucociliary regeneration and function are key players for airway defense and are impaired in chronic obstructive pulmonary disease (COPD). Using transcriptome analysis in COPD-derived bronchial biopsies, we observed a positive correlation between cilia-related genes and microRNA-449 (miR449). In vitro, miR449 was strongly increased during airway epithelial mucociliary differentiation. In vivo, miR449 was upregulated during recovery from chemical or infective insults. miR0449-/- mice (both alleles are deleted) showed impaired ciliated epithelial regeneration after naphthalene and Haemophilus influenzae exposure, accompanied by more inten